# Train T1 + domain classifier on Colab GPU

This offloads the two CPU-heavy parts of the fake-news-orchestration pipeline (T1 fine-tuning, the 3-encoder x 12-config domain-classifier grid search for Table 5) onto Colab's GPU. Everything LLM-dependent (T2, T3, the router) stays on your laptop since it talks to your local LM Studio server, which Colab can't reach.

**Before running:** in Colab's menu, go to `Runtime > Change runtime type` and select **T4 GPU**.

**Steps:**
1. Run the cells in order.
2. When prompted, upload `fake-news-orchestration.zip` (the project code + AMTCele data, no `.venv`).
3. At the end, download `checkpoints.zip` and unzip it into your local project so it becomes `fake-news-orchestration/checkpoints/t1/...` and `fake-news-orchestration/checkpoints/domain_classifier/...`.
4. Locally, `configs/default.yaml` already points at those same paths (`paths.t1_checkpoint_dir`, `paths.domain_classifier_checkpoint_dir`) -- once the files are there, every local script that supports it (`run_catalogue`, `run_main_veracity`, `run_ablations`, `run_routing_analysis`, `run_efficiency`, `run_domain_classification`) will auto-detect and load the checkpoint instead of retraining.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -- go set Runtime > Change runtime type > T4 GPU')

## Upload the project
Upload `fake-news-orchestration.zip` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/project')
%cd /content/project

## Install dependencies
Reuses the project's own `requirements.txt` so versions match your local venv. `openai` (API-only extra) and `bitsandbytes` aren't needed here -- nothing in this notebook calls an LLM or does local 4-bit inference.

In [ ]:
!pip install -q sentence-transformers rank-bm25 pyyaml faiss-cpu dateparser

## Train
Trains T1 once per seed in `configs/default.yaml`'s `seeds` list (reused by Table 2/6/8/10/11's scripts), then runs the full domain-classifier grid search for Table 5 at the first seed. Prints progress per epoch / per grid config, same as the local logging.

In [ ]:
from colab.train_on_colab import main
main()

## Package checkpoints for download
Checkpoints can be a few GB total (3 T1 seeds + 3 domain-classifier encoders), so this saves to Google Drive first (more reliable than a direct browser download for large files) and also offers a direct download.

In [ ]:
!zip -qr checkpoints.zip checkpoints/
import os
print('checkpoints.zip size (MB):', round(os.path.getsize('checkpoints.zip') / 1e6))

In [ ]:
# Option A: save to Google Drive (recommended for multi-GB files)
from google.colab import drive
drive.mount('/content/drive')
!cp checkpoints.zip /content/drive/MyDrive/checkpoints.zip
print('Saved to Google Drive: MyDrive/checkpoints.zip')

In [ ]:
# Option B: direct browser download (can be flaky for very large files)
from google.colab import files
files.download('checkpoints.zip')